<a href="https://colab.research.google.com/github/Badhon63/-Validation-of-a-Mathematical-Expression./blob/main/09-image-classification-cnn/09_image_classification_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Machine Learning Lab**

# 10. Image Classification Using CNN

This notebook trains a **Convolutional Neural Network (CNN)** to classify images of chess pieces into their type (king, queen, pawn, etc.). CNNs are the standard architecture for image tasks: instead of flattening an image into raw pixels right away, they slide small filters across the image to learn local patterns (edges, textures, shapes) and build them up into higher-level features layer by layer. We'll first build and train a CNN from scratch, then compare it against **transfer learning** — reusing a network (VGG16) that was already pretrained on millions of general images, and just fine-tuning it for our chess pieces.

**Note:** this notebook trains a real CNN on real images and works much better with a GPU runtime (in Colab: Runtime → Change runtime type → GPU). On CPU it will be slow.

Import TensorFlow/Keras layers and utilities we'll use throughout.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Rescaling, Input
from tensorflow.keras.optimizers import Adam

## Downloading the Dataset

This dataset comes from [Kaggle](https://www.kaggle.com/datasets/anshulmehtakaggl/chess-pieces-detection-images-dataset), so it's downloaded via the Kaggle API rather than uploaded manually. To do this, you need a Kaggle API token:

1. Log into Kaggle, go to your [account settings](https://www.kaggle.com/settings).
2. Under "API", click "Create New Token" — this downloads a file called `kaggle.json`.
3. Run the cell below and upload that `kaggle.json` file when prompted.

(The original version of this notebook mounted Google Drive here, but nothing in the notebook actually read from Drive — the dataset always came from this Kaggle download instead — so that step has been replaced with the Kaggle credential upload it actually needs.)

In [3]:
!pip install -q kaggle

from google.colab import files
print("Upload your kaggle.json (Kaggle account -> Settings -> API -> Create New Token)")
uploaded = files.upload()  # select kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

ERROR: Operation cancelled by user
Upload your kaggle.json (Kaggle account -> Settings -> API -> Create New Token)


KeyboardInterrupt: 

Download and unzip the chess pieces dataset using the authenticated Kaggle CLI.

In [5]:
!kaggle datasets download -d anshulmehtakaggl/chess-pieces-detection-images-dataset -p . --unzip -q
!mv 'Chess Pieces Detection Images Dataset' Chess 2>/dev/null || true

Dataset URL: https://www.kaggle.com/datasets/anshulmehtakaggl/chess-pieces-detection-images-dataset
License(s): CC0-1.0


Set up the basic parameters for loading the images: where they live on disk, what size to resize them to, and how many images per training batch.

In [6]:
# Path to chess piece images
dataset_path = 'Chess'
img_size = (224, 224)
batch_size = 32
seed = 42

`image_dataset_from_directory` scans `dataset_path` and automatically treats each subfolder as a separate class — it infers labels straight from the folder structure.

In [7]:
# Load full dataset
full_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    image_size=img_size,
    batch_size=batch_size,
    seed=seed,
    shuffle=True  # Shuffle before splitting
)

Found 651 files belonging to 5 classes.


Manually split the batches into 80% for training and 20% for testing.

In [9]:
# Split manually: 80% train, 20% test
dataset_size = len(full_ds)
train_size = int(0.8 * dataset_size)
test_size = dataset_size - train_size

train_ds = full_ds.take(train_size)
test_ds = full_ds.skip(train_size)

## Data Augmentation

With a modest number of training images, **data augmentation** helps the model generalize: it applies random, realistic transformations (flip, rotate, zoom, brightness/contrast changes) to each image every time it's seen during training, so the model effectively sees a slightly different version of each photo on every epoch instead of memorizing the exact pixels.

In [10]:
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, RandomBrightness, RandomContrast

data_augmentation = tf.keras.Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.2),
    RandomZoom(0.2),
    RandomBrightness(0.2),
    RandomContrast(0.2),
])

Apply the augmentation to the training set only (never to the test set, since we want to evaluate on realistic, unaltered images).

In [11]:
# Apply augmentation
augmented_train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

Visualize a grid of augmented training images along with their class labels, to sanity-check the augmentation looks reasonable.

In [ ]:
# Show Images

import matplotlib.pyplot as plt

# Take a single batch of images from the augmented training dataset
image_batch, label_batch = next(iter(augmented_train_ds))

# Number of images to display (25 in a 5x5 grid)
num_images_to_display = 25

# Create a 5x5 grid for the images
fig, axes = plt.subplots(5, 5, figsize=(10, 10))

# Iterate over the images in the batch
for i in range(num_images_to_display):
  ax = axes[i // 5, i % 5]  # Get the current subplot
  ax.imshow(image_batch[i].numpy().astype("uint8"))  # Display the image
  ax.set_title(f"Class: {label_batch[i].numpy()}")  # Set the class label as the title
  ax.axis("off")  # Turn off axes

plt.tight_layout()
plt.show()

`prefetch` overlaps data loading with model training (loads the next batch while the current one trains), which speeds things up.

In [ ]:
# Prefetch
AUTOTUNE = tf.data.AUTOTUNE
augmented_train_ds = augmented_train_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

## Building a Custom CNN

Now we define the CNN architecture. It stacks three `Conv2D` + `MaxPooling2D` blocks (each block detects increasingly complex patterns and shrinks the spatial size), followed by `Dense` classification layers with `Dropout` in between to reduce overfitting. `Rescaling(1./255)` normalizes pixel values from [0, 255] to [0, 1] right inside the model, so we don't have to remember to do it separately every time we use the model.

In [ ]:
# Build Custom CNN model
model = Sequential([
    Input(shape=(img_size[0], img_size[1], 3), name='input_layer'),
    Rescaling(1./255),

    Conv2D(32, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(64, (5, 5), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(128, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Dropout(0.4),

    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(full_ds.class_names), activation='softmax')
])

# Compile
model.compile(optimizer='Adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Train the custom CNN for 10 epochs.

In [ ]:
# Train
history = model.fit(augmented_train_ds, epochs=10)

Evaluate on the held-out test set.

In [ ]:
# Test
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc:.2f}")

## Transfer Learning with VGG16

Instead of training a CNN from scratch, we can start from **VGG16**, a network already trained on the huge ImageNet dataset (1.4 million general images, 1000 categories). Its early and middle layers have already learned generic visual features (edges, textures, shapes) that transfer well to new image tasks. We load VGG16 without its original classification head (`include_top=False`), freeze its weights (`base_model.trainable = False`, so its already-learned features aren't disturbed by our small dataset), and attach a small new classifier on top for our chess piece classes.

We also pass `validation_data=test_ds` to `model.fit()` this time (missing in an earlier version of this notebook), since the accuracy plot below needs the `val_accuracy` history that only gets recorded when validation data is supplied during training.

In [ ]:
# Transfer Learning

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

# Load pre-trained VGG16 model (excluding top classification layers)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(img_size[0], img_size[1], 3))

# Freeze the base model layers
base_model.trainable = False

# Create a new model on top of the base model
model = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dense(len(full_ds.class_names), activation='softmax')
])


# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# Train the model
history = model.fit(augmented_train_ds, epochs=10, validation_data=test_ds)

# Evaluate the model
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc:.2f}")

# Plot accuracy
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.legend()
plt.show()

## Practice Problems

1. Compare the final test accuracy of the custom CNN (trained from scratch) against the VGG16 transfer-learning model. Which performs better, and by how much? Why might that be, given how few chess piece images we have compared to ImageNet's millions?
2. Set `base_model.trainable = True` for the last few layers of VGG16 only (a technique called "fine-tuning"), then retrain with a low learning rate (e.g. `Adam(learning_rate=1e-5)`). Does accuracy improve further?
3. Add `validation_data=test_ds` to the custom CNN's `model.fit()` call too (section "Building a Custom CNN"), then plot its train vs. validation accuracy the same way as the VGG16 section. Does the gap between train and validation accuracy suggest the custom CNN is overfitting?
4. Try a different pretrained base model, such as `MobileNetV2` or `ResNet50` (both available under `tensorflow.keras.applications`), in place of VGG16. How does its accuracy and training speed compare?
5. Look at a handful of the model's mistakes: predict on `test_ds` with `model.predict()`, find images where the predicted class differs from the true label, and display them alongside both labels. Do the model's mistakes look like genuinely ambiguous chess pieces?